In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')

# Cargar el dataset maestro
df = pd.read_csv('acb_master_playbyplay.csv')

# Filtramos solo los eventos que son tiros
df_shots = df[df['shot_type'].notna()].copy()

# Asignamos el valor en puntos de cada tipo de tiro
df_shots['shot_value'] = df_shots['shot_type'].map({'3PT': 3, '2PT': 2, 'FT': 1})

# Calculamos los puntos reales conseguidos en cada acción
df_shots['points_scored'] = df_shots['shot_value'] * df_shots['shot_made']

print("Preparación de métricas base completada.")

In [ ]:
# Agrupamos por equipo para ver quién rentabiliza mejor sus tiros
team_eff = df_shots.groupby('team_actual_name').agg(
    Tiros_Totales=('shot_made', 'count'),
    Tiros_Anotados=('shot_made', 'sum'),
    Puntos_Totales=('points_scored', 'sum')
).reset_index()

# Excluimos posibles valores nulos o equipos con muestras irreales
team_eff = team_eff[team_eff['Tiros_Totales'] > 500]

# Métrica clave: PPS (Puntos Totales / Tiros Totales)
team_eff['PPS'] = round(team_eff['Puntos_Totales'] / team_eff['Tiros_Totales'], 3)
team_eff['%_Acierto'] = round((team_eff['Tiros_Anotados'] / team_eff['Tiros_Totales']) * 100, 1)

# Ordenar por el mejor Points Per Shot
team_eff = team_eff.sort_values('PPS', ascending=False).reset_index(drop=True)

print("--- TOP 10 EQUIPOS MÁS EFICIENTES (PPS) ---")
team_eff[['team_actual_name', 'PPS', '%_Acierto', 'Tiros_Totales']].head(10)

In [ ]:
# Hacemos el mismo cálculo pero a nivel individual
player_eff = df_shots.groupby('player_name_full').agg(
    Equipo=('team_abbrev_name', 'first'),
    Tiros_Totales=('shot_made', 'count'),
    Puntos_Totales=('points_scored', 'sum')
).reset_index()

# Filtro fundamental: Mínimo de 100 tiros en la temporada para evitar sesgos (ej: un jugador que tira 1 vez, anota un triple y tiene un PPS de 3.0)
player_eff = player_eff[player_eff['Tiros_Totales'] >= 100]

player_eff['PPS'] = round(player_eff['Puntos_Totales'] / player_eff['Tiros_Totales'], 3)
player_eff = player_eff.sort_values('PPS', ascending=False).reset_index(drop=True)

print("--- TOP 15 JUGADORES MÁS EFICIENTES (Mín. 100 tiros) ---")
player_eff.head(15)

In [ ]:
# Visualizamos el Top 15 de jugadores
top_players = player_eff.head(15)

plt.figure(figsize=(12, 7))
sns.barplot(x='PPS', y='player_name_full', data=top_players, palette='coolwarm')

for i, v in enumerate(top_players['PPS']):
    plt.text(v + 0.01, i, str(v), color='white', va='center', fontweight='bold')

plt.title('Top 15 Jugadores Más Eficientes de la ACB (Points Per Shot)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Points Per Shot (PPS)', fontsize=12)
plt.ylabel('Jugador', fontsize=12)
plt.xlim(0.8, top_players['PPS'].max() + 0.15) 
plt.show()